In [ ]:
import subprocess
import os
import logging
import numpy as np

import sys
sys.path.append("work/SIRF-Contribs/src/notebooks/AIRBI-MRI-recon/")
from stgeorges_utils import change_ismrmrd, to_dicom_folder, LogfileCallback

from sirf.Gadgetron import AcquisitionData, ImageData
from sirf.Gadgetron import AcquisitionModel
from sirf.Gadgetron import AcquisitionDataProcessor
from sirf.Gadgetron import CartesianGRAPPAReconstructor, FullySampledReconstructor
from sirf.Gadgetron import CoilSensitivityData
from sirf.Gadgetron import preprocess_acquisition_data

from cil.optimisation.functions import LeastSquares
from cil.optimisation.functions import ZeroFunction
from cil.optimisation.algorithms import FISTA, CGLS, GD
from cil.plugins.ccpi_regularisation.functions import FGP_TV
from cil.framework import DataContainer as cilDataContainer
from cil.optimisation.operators import LinearOperator
from cil.optimisation.utilities.callbacks import ProgressCallback, TextProgressCallback
import tempfile

from cil.optimisation.functions import L1Sparsity
from cil.optimisation.operators import WaveletOperator
from AbsFunction import FunctionOfAbs


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

command = "siemens_to_ismrmrd"

input_files = ["/home/jovyan/work/person2/person2/meas_MID00595_FID133061_pd_tse_fs_cor_uflex_no_spine.dat",
               "/home/jovyan/work/person2/person2/meas_MID00598_FID133064_pd_tse_fs_cor_uflex_no_spine.dat"]

#input_files = ["/home/jovyan/work/data/meas_MID00614_FID129152_CONVENTIONAL_RECON_SEQD_GF2_AX_RL.dat",
#               "/home/jovyan/work/data/meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.dat"]

proc_dir = tempfile.mkdtemp(prefix="stgeorges_proc_")

output_files = []
for fname in input_files:
    logger.info(f"Processing file {fname}...")
    file_in = fname
    file_out = os.path.join(proc_dir, os.path.basename(file_in).replace(".dat", ".h5"))
    if os.path.exists(file_out):
        logger.warning(f"Output file {file_out} already exists. Removing it.")
        os.remove(file_out)
    
    out = subprocess.run(
        [command, "-f", file_in, "-o", file_out, "-z", "2", "-M"],
        capture_output=True,
        text=True,
    )
    
    logger.info(out.stdout)
    logger.error(out.stderr)
    
    # Change ISMRMRD file if needed
    file_out_mod = file_out.replace(".h5", "_mod.h5")
    change_ismrmrd(file_out, file_out_mod)
    logger.info(f"Modified ISMRMRD file saved as {file_out_mod}")
    output_files.append(file_out_mod)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2)
for idx, fname in enumerate(output_files):
    acq_data = AcquisitionData(fname)
    ky_index = np.unique(acq_data.get_ISMRMRD_info('kspace_encode_step_1'))
    sampling_mask = np.zeros((np.max(ky_index)+1,np.max(ky_index)+1))
    sampling_mask[ky_index,:] = 1
    n_slices = np.unique(acq_data.get_ISMRMRD_info('slice'))
    print(f'{fname}: {len(n_slices)} slices {len(ky_index)} ky-points: {ky_index}')
    ax[idx].imshow(sampling_mask, vmin=0, vmax=1)
     


        

In [ ]:
from collections.abc import Sequence
def gaussian_variable_density_samples(
        shape: Sequence[int], low: int, high: int, fwhm: float = float('inf'), always_sample: Sequence[int] = ()
    ) -> np.ndarray:
        """Generate Gaussian variable density samples.
        Generates indices in [low, high[ with a gaussian weighting.
        Parameters
        ----------
        shape
            Shape of the output tensor. The generated indices are 1D and in the last dimension.
            All other dimensions are batch dimensions.
        low
            Lower bound of the sampling domain.
        high
            Upper bound of the sampling domain.
        fwhm
            Full-width at half-maximum of the Gaussian.
        always_sample
            indices that should always included in the samples.
            For example, `range(-n_center//2, n_center//2)`
        Returns
        -------
            1D array of selected indices.
        """
        *n_batch, n_samples = shape
        if n_samples > high - low:
            raise ValueError('n_samples must be <= (high - low)')
        n_random = n_samples - len(always_sample)
        if n_random < 0:
            raise ValueError('more always sampled points requested than total number of samples')
        elif n_random == 0:
            return np.sort(np.broadcast_to(np.array(always_sample), (*n_batch, -1)))

        pdf = np.exp(-np.log(2.0) * (2 * np.arange(low, high) / fwhm) ** 2)
        pdf[[x - low for x in always_sample]] = 0

        if len(shape) > 1:
            pdf_batch = np.broadcast_to(pdf, (*n_batch, len(pdf))).reshape(-1, len(pdf))
            # normalize each row to a valid probability distribution
            pdf_batch = pdf_batch / pdf_batch.sum(axis=-1, keepdims=True)
            idx_rand = np.array([
                np.random.default_rng().choice(len(pdf), size=n_random, replace=False, p=row) + low
                for row in pdf_batch
            ]).reshape(*n_batch, n_random)
        else:
            pdf = pdf / pdf.sum()
            idx_rand = np.random.default_rng().choice(len(pdf), size=n_random, replace=False, p=pdf) + low

        idx_always = np.broadcast_to(np.array(always_sample), (*n_batch, len(always_sample)))
        return np.sort(np.concatenate([idx_rand, idx_always], axis=-1), axis=-1)

In [ ]:
acq_data = AcquisitionData(output_files[0])
acq_data = preprocess_acquisition_data(acq_data)
ky_index = np.unique(acq_data.get_ISMRMRD_info('kspace_encode_step_1'))

In [ ]:
acceleration_factor = 3.5

nky = int(np.max(ky_index)+1)
ky_index_gauss = gaussian_variable_density_samples([1,int(len(ky_index)/acceleration_factor)], -nky//2, nky//2-1, nky*0.8, range(-13, 13)) + nky//2
sampling_mask = np.zeros((nky, nky))
sampling_mask[ky_index_gauss,:] = 1

plt.figure()
plt.imshow(sampling_mask, vmin=0, vmax=1)

In [ ]:
# Reconstruct fully sampled data
csm = CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data)
E = AcquisitionModel(acqs=acq_data, imgs=csm)
E.set_coil_sensitivity_maps(csm)
image_data = E.inverse(acq_data)


# Create undersampled data
us_index = []
for ky_idx in acq_data.get_ISMRMRD_info('kspace_encode_step_1'):
    if ky_idx in ky_index_gauss:
        us_index.append(ky_idx)
acq_data_us = acq_data.get_subset(us_index)
E = AcquisitionModel(acqs=acq_data_us, imgs=csm)
E.set_coil_sensitivity_maps(csm)
image_data_us = E.inverse(acq_data_us)


# Reconstruct AI undersampled data
acq_data_ai = AcquisitionData(output_files[1])
acq_data_ai = preprocess_acquisition_data(acq_data_ai)
csm = CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data_ai)
E = AcquisitionModel(acqs=acq_data_ai, imgs=csm)
E.set_coil_sensitivity_maps(csm)
image_data_ai = E.inverse(acq_data_ai)


In [ ]:
# Visualise images
imgs = [image_data, image_data_us, image_data_ai]

fig, ax = plt.subplots(2,3, figsize=(12,8))

for idx, img in enumerate(imgs):
    image_array = img.as_array()
    image_array[np.isnan(image_array)] = 0
    image_array = image_array/abs(image_array).max()
    
    centre_slice = image_array.shape[0]//2
    ax[0,idx].imshow(abs(image_array[centre_slice,:,:]), vmin=0, vmax=0.6, cmap='gray')
    ax[1,idx].imshow(np.angle(image_array[centre_slice,:,:]), vmin=-np.pi, vmax=np.pi, cmap='bwr')